# News Reliability Assessment - Baseline Training

Notebook này dùng để train baseline ML trên Google Colab và tải artifacts về máy.

Mặc định dùng VFND để bám sát bài toán fake/real news. Nếu muốn thí nghiệm thêm ViFactCheck, thêm `%env INCLUDE_VIFACTCHECK=1` trước cell download/train.

In [1]:
!git clone https://github.com/imnothoan/doAnChuyenNganh1.git
%cd doAnChuyenNganh1

Cloning into 'doAnChuyenNganh1'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 202 (delta 69), reused 177 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 807.43 KiB | 2.82 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/doAnChuyenNganh1


In [2]:
!python3 -m pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 117.8 MB/s eta 0:00:00


In [3]:
!python3 scripts/download_data.py
!python3 scripts/prepare_data.py
!python3 scripts/train_baseline.py
!python3 scripts/evaluate.py

2026-05-15 17:25:13,678 | INFO | Trying source for VFND: https://github.com/VFND/VFND-vietnamese-fake-news-datasets
2026-05-15 17:25:14,506 | INFO | Trying source for TALLIP: https://github.com/Arko98/TALLIP-FakeNews-Dataset
2026-05-15 17:25:14,966 | WARNING | Failed to download TALLIP: Repository cloned, but data files are distributed through the TALLIP zip link in README.
[ok] VFND: downloaded
[manual_required] TALLIP: Repository cloned, but data files are distributed through the TALLIP zip link in README.
[manual_required] Zenodo: Zenodo latest endpoint may require browser/manual download
[skipped] ViFactCheck: optional_fact_checking_dataset_set_INCLUDE_VIFACTCHECK=1_to_enable
2026-05-15 17:25:14,967 | INFO | Saved source status report to reports/dataset_sources.json
2026-05-15 17:25:20,597 | INFO | Dataset prepared: train=350 val=75 test=75
2026-05-15 17:25:23,197 | INFO | Training lr model
2026-05-15 17:25:25,529 | INFO | Training svm model
2026-05-15 17:25:27,311 | INFO | Trainin

In [4]:
from pathlib import Path
import json

print(Path('reports/model_comparison.md').read_text())
metadata = json.loads(Path('models/reports/model_metadata.json').read_text())
print(json.dumps(metadata, indent=2, ensure_ascii=False))

# Model Comparison

Best model selected by validation F1 macro: **svm**.

Label convention: `0 = reliable/real`, `1 = unreliable/fake/clickbait`.

| Model | Val Acc | Val F1 Macro | Test Acc | Test F1 Macro | Test ROC-AUC |
|---|---:|---:|---:|---:|---:|
| lr | 0.9200 | 0.9196 | 0.8933 | 0.8929 | 0.9772 |
| svm | 0.9467 | 0.9466 | 0.9067 | 0.9064 | 0.9879 |
| rf | 0.9200 | 0.9196 | 0.8933 | 0.8929 | 0.9744 |
| nb | 0.9067 | 0.9061 | 0.8933 | 0.8929 | 0.9744 |
{
  "best_model": "svm",
  "label_convention": {
    "0": "reliable/real",
    "1": "unreliable/fake/clickbait"
  },
  "class_order": [
    0,
    1
  ],
  "dataset_sizes": {
    "train": 350,
    "validation": 75,
    "test": 75
  },
  "best_model_test_after_refit": {
    "accuracy": 0.92,
    "precision_macro": 0.9214285714285715,
    "recall_macro": 0.9196301564722618,
    "f1_macro": 0.9198717948717949,
    "precision_weighted": 0.9211428571428572,
    "recall_weighted": 0.92,
    "f1_weighted": 0.91991452991453,
    "roc_auc"

In [5]:
import joblib
from src.models.inference import predict_reliability

model = joblib.load('models/artifacts/baseline_best.joblib')
sample = 'Tin đồn chưa kiểm chứng đang lan truyền gây hoang mang trên mạng xã hội.'
predict_reliability(sample, model, model_name='Best model')

{'id': 'f068094d0babf638',
 'text': 'Tin đồn chưa kiểm chứng đang lan truyền gây hoang mang trên mạng xã hội.',
 'model_name': 'Best model',
 'predicted_label': 1,
 'model_predicted_label': 0,
 'label_name': 'unreliable',
 'label_vi': 'Nghi ngờ',
 'label_description': 'Nội dung có dấu hiệu tin giả, clickbait hoặc thiếu độ tin cậy.',
 'confidence': 0.786125,
 'risk_score': 0.786125,
 'probabilities': {'reliable': 0.21387500000000004, 'unreliable': 0.786125},
 'model_probabilities': {'reliable': 0.5777169137865901,
  'unreliable': 0.4222830862134099},
 'lexical_risk_score': 0.786125,
 'suspicious_terms': [{'term': 'chưa kiểm chứng',
   'category': 'credibility',
   'count': 1},
  {'term': 'hoang mang', 'category': 'emotion', 'count': 1},
  {'term': 'lan truyền', 'category': 'credibility', 'count': 1},
  {'term': 'tin đồn', 'category': 'credibility', 'count': 1}],
 'text_stats': {'characters': 72,
  'words': 15,
  'sentences': 1,
  'exclamation_marks': 0,
  'question_marks': 0,
  'upperca

In [6]:
from google.colab import files

for path in [
    'models/artifacts/baseline_best.joblib',
    'reports/model_comparison.md',
    'reports/metrics_baseline.json',
    'models/reports/model_metadata.json',
]:
    files.download(path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>